# POEM Pulse88 continuation training

Attach the POEM code dataset and the pre-tokenized Pulse88 dataset at `/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenized-500k`. Set the accelerator to **GPU T4 x2**. This notebook trains the 40M-ish flat-token continuation models in order: `G_CONT` first, then `H_CONT`, using the frozen `CustomDeltaTokenizer` vocabulary.

In [ ]:
!nvidia-smi
!python --version
!pip install -q pretty_midi mido huggingface_hub numpy

In [ ]:
from pathlib import Path
import os, shutil

def first_existing(candidates):
    for path in candidates:
        p = Path(path)
        if p.exists():
            return p
    return None

def find_repo_dir(input_root):
    explicit = first_existing([
        input_root / 'poem-base',
        input_root / 'POEM-BASE',
        input_root / 'poem',
        input_root / 'POEM',
        input_root / 'datasets' / 'chickaboomcmurtrie' / 'poem-base',
    ])
    if explicit is not None:
        return explicit
    matches = []
    for p in input_root.rglob('*'):
        if p.is_dir() and (p / 'train_continuation.py').exists() and (p / 'models').exists():
            matches.append(p)
    if not matches:
        raise FileNotFoundError('Could not find POEM repo dataset containing train_continuation.py.')
    return sorted(matches, key=lambda p: len(p.parts))[0]

INPUT = Path('/kaggle/input')
repo_src = find_repo_dir(INPUT)
repo_dst = Path('/kaggle/working/POEM')
if repo_dst.exists():
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst, ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc'))
os.chdir(repo_dst)
print('Repo:', repo_dst)

DATA_ROOT = Path('/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenized-500k')
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
print('Pulse88:', DATA_ROOT)
print('Tokenizer spec:', Path('custom_tokenizer.json').resolve())

In [ ]:
from getpass import getpass
import os

HF_TOKEN = getpass('Paste a Hugging Face token with write access: ')
os.environ['HF_TOKEN'] = HF_TOKEN
HF_NAMESPACE = input('Hugging Face username or org: ').strip()
HF_REPO_ID = f'{HF_NAMESPACE}/POEM'
print('HF repo:', HF_REPO_ID)

In [ ]:
# 40M-ish continuation models. G_CONT trains first, then H_CONT.
VARIANTS = 'G_CONT H_CONT'
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUMULATION_STEPS = 4
SEED_LENGTH = 1024
CONTINUATION_LENGTH = 1024
MAX_PIECES = 0
MAX_STEPS = 0
MAX_HOURS = 11.75

cmd = f"""
python -u scripts/kaggle_train_continuation.py \
  --repo_dir /kaggle/working/POEM \
  --data_root '{DATA_ROOT}' \
  --hf_repo_id '{HF_REPO_ID}' \
  --private \
  --variants {VARIANTS} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --grad_accumulation_steps {GRAD_ACCUMULATION_STEPS} \
  --seed_length {SEED_LENGTH} \
  --continuation_length {CONTINUATION_LENGTH} \
  --max_pieces {MAX_PIECES} \
  --max_steps {MAX_STEPS} \
  --max_hours {MAX_HOURS} \
  --val_interval 1000 \
  --checkpoint_interval 2500 \
  --num_workers 2
"""
print(cmd)
!{cmd}

In [ ]:
from pathlib import Path
for path in sorted(Path('/kaggle/working/continuation_metrics').glob('*/summary.json')):
    print('\n===', path, '===')
    print(path.read_text()[:4000])